# Notebook 03 — Link MTSamples Clinical Notes to MPI

**Goal:** Assign each of the 5,000 MTSamples clinical notes a `patient_id` from the MPI.

**Linking logic:**
- Match note's `medical_specialty` to patient's `primary_condition` using a specialty-condition mapping
- If no match found → fallback to random patient assignment

**Output:** `data_preparation/linked/notes_linked.csv`

## 1. Imports & Paths

In [ ]:
import pandas as pd
import numpy as np
import random
import os

MTSAMPLES_PATH = "../raw/mtsamples/mtsamples.csv"
MPI_PATH       = "../linked/patients_master.csv"
OUTPUT_DIR     = "../linked/"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Paths OK")

## 2. Load Data

In [ ]:
mtsamples = pd.read_csv(MTSAMPLES_PATH)
mpi       = pd.read_csv(MPI_PATH)

print(f"MTSamples records : {len(mtsamples)} rows")
print(f"MPI patients      : {len(mpi)} rows")
print()
print("MTSamples columns:")
print(mtsamples.columns.tolist())
print()
print("MTSamples sample:")
mtsamples.head(3)

## 3. Explore Medical Specialties

In [ ]:
print("All medical specialties in MTSamples:")
print(mtsamples["medical_specialty"].value_counts())
print()
print("Top 10 primary conditions in MPI:")
print(mpi["primary_condition"].value_counts().head(10))

## 4. Define Specialty → Condition Mapping

In [ ]:
# Maps each MTSamples specialty to keywords found in MPI primary_condition
# If a patient's condition contains any of the keywords → they match this specialty

SPECIALTY_CONDITION_MAP = {
    "Cardiovascular / Pulmonary": [
        "heart", "cardiac", "coronary", "hypertension", "atrial",
        "congestive", "hypoxemia", "pulmonary", "asthma", "copd",
        "pneumonia", "bronchitis", "sinusitis"
    ],
    "Orthopedic": [
        "back pain", "neck pain", "osteoporosis", "fracture",
        "sprain", "arthritis", "osteoarthritis", "carpal"
    ],
    "Neurology": [
        "stroke", "epilepsy", "migraine", "anxiety", "depression",
        "dementia", "alzheimer", "neuropathy"
    ],
    "Gastroenterology": [
        "crohn", "ibs", "gerd", "gastro", "bowel", "colitis",
        "liver", "hepatitis", "appendix"
    ],
    "General Medicine": [
        "diabetes", "obesity", "prediabetes", "hyperlipidemia",
        "anemia", "hypothyroid", "vitamin"
    ],
    "Obstetrics / Gynecology": [
        "miscarriage", "pregnancy", "gynecol", "ovarian", "uterine"
    ],
    "ENT - Otolaryngology": [
        "sinusitis", "otitis", "pharyngitis", "tonsil", "ear", "nose", "throat"
    ],
    "Urology": [
        "kidney", "renal", "urinary", "bladder", "prostate"
    ],
    "Hematology - Oncology": [
        "cancer", "tumor", "leukemia", "lymphoma", "anemia"
    ],
    "Dermatology": [
        "skin", "dermatitis", "eczema", "psoriasis", "rash"
    ],
    "Ophthalmology": [
        "eye", "vision", "glaucoma", "cataract", "retinal"
    ],
    "Podiatry": [
        "foot", "ankle", "plantar", "heel"
    ],
    "Psychiatry / Psychology": [
        "anxiety", "depression", "panic", "stress", "bipolar", "schizophrenia"
    ],
    "Endocrinology": [
        "diabetes", "thyroid", "hormone", "adrenal", "prediabetes", "hyperlipidemia"
    ],
    "Infectious Disease": [
        "covid", "influenza", "viral", "bacterial", "sepsis", "infection"
    ],
}

print(f"Specialty mappings defined: {len(SPECIALTY_CONDITION_MAP)}")

## 5. Build Specialty → Patient Pool Lookup

In [ ]:
# For each specialty, find all MPI patients whose primary_condition matches
specialty_patient_pools = {}

for specialty, keywords in SPECIALTY_CONDITION_MAP.items():
    # Find patients whose condition contains any of the keywords
    mask = mpi["primary_condition"].str.lower().apply(
        lambda cond: any(kw in str(cond).lower() for kw in keywords)
    )
    matched_patients = mpi[mask]["patient_id"].tolist()
    specialty_patient_pools[specialty] = matched_patients
    print(f"{specialty:<40} → {len(matched_patients)} patients")

print()
# All patients as fallback pool
all_patient_ids = mpi["patient_id"].tolist()
print(f"Total MPI patients (fallback pool): {len(all_patient_ids)}")

## 6. Link Each Note to a Patient ID

In [ ]:
def find_patient_for_note(specialty):
    """
    Find a matching patient_id for a clinical note.
    Strategy:
      1. Look up specialty in pool → pick randomly from matched patients
      2. If specialty not in map or pool is empty → random from all patients
    Returns (patient_id, match_type)
    """
    specialty = str(specialty).strip()

    # Try specialty match
    if specialty in specialty_patient_pools:
        pool = specialty_patient_pools[specialty]
        if len(pool) > 0:
            return random.choice(pool), "specialty_match"

    # Fallback: random patient
    return random.choice(all_patient_ids), "random_fallback"


print("Linking MTSamples notes to patient IDs...")
results = mtsamples["medical_specialty"].apply(find_patient_for_note)

mtsamples["patient_id"]  = results.apply(lambda x: x[0])
mtsamples["match_type"]  = results.apply(lambda x: x[1])

print("Done.")
print()
print("Match type distribution:")
print(mtsamples["match_type"].value_counts())

## 7. Clean Up & Final Structure

In [ ]:
# Drop unnamed index column if present
mtsamples = mtsamples.loc[:, ~mtsamples.columns.str.contains("^Unnamed")]

# Reorder columns — patient_id and match_type first
cols = ["patient_id", "match_type"] + [
    c for c in mtsamples.columns if c not in ["patient_id", "match_type"]
]
mtsamples = mtsamples[cols]

print("Final columns:")
print(mtsamples.columns.tolist())
print()
mtsamples.head(3)

## 8. Quality Check

In [ ]:
print("=== Notes Linking Quality Check ===")
print(f"Total notes             : {len(mtsamples)}")
print(f"Unique patients assigned: {mtsamples['patient_id'].nunique()}")
print(f"Patients with 0 notes   : {len(mpi) - mtsamples['patient_id'].nunique()}")
print()
print("Match type distribution:")
print(mtsamples["match_type"].value_counts())
print()
print("Notes per patient (stats):")
notes_per_patient = mtsamples.groupby("patient_id").size()
print(notes_per_patient.describe())
print()
print("Match type by specialty (top 10):")
print(mtsamples.groupby(["medical_specialty", "match_type"]).size().reset_index(
    name="count").sort_values("count", ascending=False).head(10))
print()
print("Missing transcription text:")
print(mtsamples["transcription"].isna().sum())

## 9. Save Output

In [ ]:
output_path = os.path.join(OUTPUT_DIR, "notes_linked.csv")
mtsamples.to_csv(output_path, index=False)

print(f"Saved → {output_path}")
print(f"Shape  : {mtsamples.shape}")